In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from pyproj import Transformer
from sklearn.inspection import permutation_importance

In [2]:
%store

Stored variables and their in-db values:
accmag_merge_df                    ->               device             drive_id  UnixTim
cn0dbhz_merge_df                   ->                    drive_id        device  UnixTim
cn0dbhzdelta_merge_df              ->                    drive_id        device  UnixTim
cn0dbhzmean_merge_df               ->                    drive_id        device  UnixTim
elevationdeg_merge_df              ->                    drive_id        device  UnixTim
energy_merge_df                    ->               device             drive_id  UnixTim
energydelta_merge_df               ->               device             drive_id  UnixTim
energystd_merge_df                 ->               device             drive_id  UnixTim
gnss_df                            ->           utcTimeMillis          device           
gt_df                              ->         UnixTimeMillis          device            
gyromag_merge_df                   ->               device           

In [4]:
%store -r cn0dbhz_merge_df
%store -r satcount_merge_df
%store -r elevationdeg_merge_df
%store -r energy_merge_df

%store -r satcountdelta_merge_df
%store -r energydelta_merge_df
%store -r cn0dbhzdelta_merge_df
%store -r cn0dbhzmean_merge_df
%store -r energystd_merge_df

In [5]:
merge_df = cn0dbhz_merge_df.merge(satcount_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SatCount']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(elevationdeg_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SvElevationDegrees']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(energy_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'TotalMotionEnergy']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')

merge_df = merge_df.merge(satcountdelta_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SatCountDelta']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(cn0dbhzdelta_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'Cn0DbHzDelta']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(energydelta_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'EnergyDelta']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(cn0dbhzmean_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'Cn0DbHzMean']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(energystd_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'EnergyStd']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')

In [6]:
print(merge_df.isna().sum())

drive_id                  0
device                    0
UnixTimeMillis            0
Cn0DbHz                   0
ErrorXEcefMeters          0
ErrorYEcefMeters          0
ErrorZEcefMeters          0
SatCount                  0
SvElevationDegrees        1
TotalMotionEnergy     20092
SatCountDelta             0
Cn0DbHzDelta              0
EnergyDelta               0
Cn0DbHzMean               0
EnergyStd                 0
dtype: int64


In [7]:
# drop null values

merge_df = merge_df.dropna(subset=['SvElevationDegrees', 'TotalMotionEnergy'])

print(merge_df.isna().sum())
print()
print('Merge Shape: ')
print(merge_df.shape)
print()

drive_id              0
device                0
UnixTimeMillis        0
Cn0DbHz               0
ErrorXEcefMeters      0
ErrorYEcefMeters      0
ErrorZEcefMeters      0
SatCount              0
SvElevationDegrees    0
TotalMotionEnergy     0
SatCountDelta         0
Cn0DbHzDelta          0
EnergyDelta           0
Cn0DbHzMean           0
EnergyStd             0
dtype: int64

Merge Shape: 
(133538, 15)



In [10]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'TotalMotionEnergy', 'SatCountDelta', 'EnergyDelta', 'Cn0DbHzDelta', 'Cn0DbHzMean', 'EnergyStd']]
y = pd.Series(merge_df['ErrorXEcefMeters'])

In [11]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# initial hyperparameters chosen to prevent overfitting

# rf_regressor_x = RandomForestRegressor(
#     n_estimators=300,        # more trees
#     max_depth=7,            # limit depth
#     min_samples_split=3,     # prevent tiny splits
#     min_samples_leaf=2,      # ensure leaves have enough samples
#     max_features='sqrt',     # reduce correlation between trees
#     random_state=42
# )

rf_regressor_x = RandomForestRegressor(
    n_estimators=500,           # Higher count to stabilize the "averaging" with 153k rows
    max_depth=9,               # Increased from 7 to capture non-linearity, but capped
    min_samples_leaf=50,        # CRITICAL: Forces each leaf to represent a statistically significant group (approx 0.03% of your data)
    max_features=0.3,           # Try 30% of features instead of 'sqrt' to give each tree more "context" per split
    bootstrap=True,
    n_jobs=-1,                  # Use all cores for speed
    random_state=42
)

rf_regressor_x.fit(X_train, y_train)

y_train_pred = rf_regressor_x.predict(X_train)
y_test_pred = rf_regressor_x.predict(X_test)

In [13]:
# metrics for train data

print('Goodness of Fit of Model \tTrain Dataset')
print("Explained Variance (R^2) \t:", rf_regressor_x.score(X_train, y_train)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_train, y_train_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Train Dataset
Explained Variance (R^2) 	: 0.2394655718906713
Mean Squared Error (MSE) 	: 5.497521086126621
Root Mean Squared Error (RMSE) 	: 2.3446793141337308


In [14]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", rf_regressor_x.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: 0.2241093545830274
Mean Squared Error (MSE) 	: 5.609018357582926
Root Mean Squared Error (RMSE) 	: 2.368336622522847


In [15]:
result = permutation_importance(
    rf_regressor_x, X_train, y_train, n_repeats=10, random_state=42, n_jobs=-1
)

# Sort and plot
sorted_idx = result.importances_mean.argsort()
plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=X_test.columns[sorted_idx])
plt.title("Permutation Importance for rf_regressor_x (Validation Set)")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [41]:
rf_regressor_randcv = RandomForestRegressor(random_state=42)

search_space = {
    'n_estimators' : [100, 150, 200, 250, 300],
    'max_depth' : [5, 10, 15, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features' : ['sqrt', 'log2', None]
}

In [43]:
randsearch_cv = RandomizedSearchCV(
    estimator = rf_regressor_randcv,
    param_distributions = search_space,
    n_iter = 50,
    cv = 5,
    scoring = 'r2',
    n_jobs = -1,
    verbose = 2,
    random_state = 42
)

randsearch_cv.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END max_depth=10, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time= 4.6min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=  28.4s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 3.6min


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  57.1s
[CV] END max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=10, n_estimators=150; total time= 3.7min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=  31.4s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 3.5min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  56.7s
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  56.6s
[CV] END max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=10, n_estimators=150; total time= 3.7min
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 3.5min
[CV] END max_depth=5, max_feat

,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'max_depth': [5, 10, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,50
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


[CV] END max_depth=None, max_features=None, min_samples_leaf=4, min_samples_split=2, n_estimators=150; total time= 3.3min
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  36.2s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time= 2.0min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=200; total time= 1.2min
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=300; total time= 3.1min
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time=  20.7s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time=  20.4s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time= 1.2min
[CV] END max_depth=15, max_featur

In [45]:
print("Best params:", randsearch_cv.best_params_)

Best params: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': None}


In [69]:
modelx = randsearch_cv.best_estimator_

y_test_pred = modelx.predict(X_test)

In [71]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", modelx.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: 0.2991634540649919
Mean Squared Error (MSE) 	: 4.943087685201938
Root Mean Squared Error (RMSE) 	: 2.2233055762089786


In [16]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'TotalMotionEnergy', 'SatCountDelta', 'EnergyDelta', 'Cn0DbHzDelta', 'Cn0DbHzMean', 'EnergyStd']]
y = pd.Series(merge_df['ErrorYEcefMeters'])

In [17]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
# initial hyperparameters chosen to prevent overfitting

# rf_regressor_y = RandomForestRegressor(
#     n_estimators=300,        # more trees
#     max_depth=7,            # limit depth
#     min_samples_split=3,     # prevent tiny splits
#     min_samples_leaf=2,      # ensure leaves have enough samples
#     max_features='sqrt',     # reduce correlation between trees
#     random_state=42
# )

rf_regressor_y = RandomForestRegressor(
    n_estimators=500,           # Higher count to stabilize the "averaging" with 153k rows
    max_depth=9,               # Increased from 7 to capture non-linearity, but capped
    min_samples_leaf=50,        # CRITICAL: Forces each leaf to represent a statistically significant group (approx 0.03% of your data)
    max_features=0.3,           # Try 30% of features instead of 'sqrt' to give each tree more "context" per split
    bootstrap=True,
    n_jobs=-1,                  # Use all cores for speed
    random_state=42
)

rf_regressor_y.fit(X_train, y_train)

y_train_pred = rf_regressor_y.predict(X_train)
y_test_pred = rf_regressor_y.predict(X_test)

In [19]:
# metrics for train data

print('Goodness of Fit of Model \tTrain Dataset')
print("Explained Variance (R^2) \t:", rf_regressor_y.score(X_train, y_train)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_train, y_train_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Train Dataset
Explained Variance (R^2) 	: 0.22658757827331177
Mean Squared Error (MSE) 	: 8.61510071652542
Root Mean Squared Error (RMSE) 	: 2.9351491813067048


In [20]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", rf_regressor_y.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: 0.21162110810433588
Mean Squared Error (MSE) 	: 8.751959433149635
Root Mean Squared Error (RMSE) 	: 2.9583710776624415


In [21]:
result = permutation_importance(
    rf_regressor_y, X_train, y_train, n_repeats=10, random_state=42, n_jobs=-1
)

# Sort and plot
sorted_idx = result.importances_mean.argsort()
plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=X_test.columns[sorted_idx])
plt.title("Permutation Importance for rf_regressor_y (Validation Set)")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [77]:
rf_regressor_randcv = RandomForestRegressor(random_state=42)

search_space = {
    'n_estimators' : [100, 150, 200, 250, 300],
    'max_depth' : [5, 10, 15, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features' : ['sqrt', 'log2', None]
}

In [78]:
randsearch_cv = RandomizedSearchCV(
    estimator = rf_regressor_randcv,
    param_distributions = search_space,
    n_iter = 50,
    cv = 5,
    scoring = 'r2',
    n_jobs = -1,
    verbose = 2,
    random_state = 42
)

randsearch_cv.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END max_depth=10, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time= 3.9min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=  28.9s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 3.4min


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  46.1s
[CV] END max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=10, n_estimators=150; total time= 3.4min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=  29.5s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 3.3min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  45.9s
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  44.6s
[CV] END max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=10, n_estimators=150; total time= 3.4min
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 3.3min
[CV] END max_depth=10, max_fea

,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'max_depth': [5, 10, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,50
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [79]:
print("Best params:", randsearch_cv.best_params_)

Best params: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': None}


In [80]:
modely = randsearch_cv.best_estimator_

y_test_pred = modely.predict(X_test)

In [81]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", modelx.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: -0.03734148713333396
Mean Squared Error (MSE) 	: 7.717814821261508
Root Mean Squared Error (RMSE) 	: 2.778095538541018


In [24]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'TotalMotionEnergy', 'SatCountDelta', 'EnergyDelta', 'Cn0DbHzDelta', 'Cn0DbHzMean', 'EnergyStd']]
y = pd.Series(merge_df['ErrorZEcefMeters'])

In [25]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
# initial hyperparameters chosen to prevent overfitting

# rf_regressor_z = RandomForestRegressor(
#     n_estimators=300,        # more trees
#     max_depth=7,            # limit depth
#     min_samples_split=3,     # prevent tiny splits
#     min_samples_leaf=2,      # ensure leaves have enough samples
#     max_features='sqrt',     # reduce correlation between trees
#     random_state=42
# )

rf_regressor_z = RandomForestRegressor(
    n_estimators=500,           # Higher count to stabilize the "averaging" with 153k rows
    max_depth=9,               # Increased from 7 to capture non-linearity, but capped
    min_samples_leaf=50,        # CRITICAL: Forces each leaf to represent a statistically significant group (approx 0.03% of your data)
    max_features=0.3,           # Try 30% of features instead of 'sqrt' to give each tree more "context" per split
    bootstrap=True,
    n_jobs=-1,                  # Use all cores for speed
    random_state=42
)

rf_regressor_z.fit(X_train, y_train)

y_train_pred = rf_regressor_z.predict(X_train)
y_test_pred = rf_regressor_z.predict(X_test)

In [27]:
# metrics for train data

print('Goodness of Fit of Model \tTrain Dataset')
print("Explained Variance (R^2) \t:", rf_regressor_z.score(X_train, y_train)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_train, y_train_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Train Dataset
Explained Variance (R^2) 	: 0.2848087802649929
Mean Squared Error (MSE) 	: 7.612033748948413
Root Mean Squared Error (RMSE) 	: 2.758991436911034


In [28]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", rf_regressor_z.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: 0.27147285456224923
Mean Squared Error (MSE) 	: 7.855538330530588
Root Mean Squared Error (RMSE) 	: 2.802773328425006


In [29]:
result = permutation_importance(
    rf_regressor_z, X_train, y_train, n_repeats=10, random_state=42, n_jobs=-1
)

# Sort and plot
sorted_idx = result.importances_mean.argsort()
plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=X_test.columns[sorted_idx])
plt.title("Permutation Importance for rf_regressor_z (Validation Set)")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [91]:
rf_regressor_randcv = RandomForestRegressor(random_state=42)

search_space = {
    'n_estimators' : [100, 150, 200, 250, 300],
    'max_depth' : [5, 10, 15, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features' : ['sqrt', 'log2', None]
}

In [92]:
randsearch_cv = RandomizedSearchCV(
    estimator = rf_regressor_randcv,
    param_distributions = search_space,
    n_iter = 50,
    cv = 5,
    scoring = 'r2',
    n_jobs = -1,
    verbose = 2,
    random_state = 42
)

randsearch_cv.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  39.0s
[CV] END max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=10, n_estimators=150; total time= 2.9min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=  23.7s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 2.8min


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END max_depth=10, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time= 3.4min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=  23.1s
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=  23.7s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 2.8min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  38.8s
[CV] END max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=250; total time=  39.4s
[CV] END max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=10, n_estimators=150; total time= 2.9min
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 2.8min
[CV] END max_depth=5, max_feature

,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'max_depth': [5, 10, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,50
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


[CV] END max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=10, n_estimators=300; total time= 7.1min
[CV] END max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=150; total time=  54.9s
[CV] END max_depth=10, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=250; total time= 3.3min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  32.8s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  33.4s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=200; total time= 2.6min
[CV] END max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=2, n_estimators=300; total time= 1.1min
[CV] END max_depth=15, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time= 3.7min
[CV] END max_depth=None, max_featur

In [93]:
print("Best params:", randsearch_cv.best_params_)

Best params: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': None}


In [94]:
modelz = randsearch_cv.best_estimator_

y_test_pred = modelz.predict(X_test)

In [95]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", modelx.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

mse = mean_squared_error(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: -0.4787128662107316
Mean Squared Error (MSE) 	: 7.153656286612744
Root Mean Squared Error (RMSE) 	: 2.6746319908751452


In [31]:
%store -r merge_test_df

In [100]:
features_test_df = merge_test_df.drop(columns=['UnixTimeMillis', 'tripId', 'WlsPositionXEcefMeters','WlsPositionYEcefMeters', 'WlsPositionZEcefMeters'])

predicted_errorx = modelx.predict(features_test_df)
predicted_errory = modely.predict(features_test_df)
predicted_errorz = modelz.predict(features_test_df)

print('predicted_errorx: ')
print(predicted_errorx)
print()
print('predicted_errory: ')
print(predicted_errory)
print()
print('predicted_errorz: ')
print(predicted_errorz)
print()

predicted_errorx: 
[-0.26498297  0.36556836 -0.7475269  ... -1.07749698 -1.29362037
 -1.03994628]

predicted_errory: 
[ 0.2118549   0.7403426  -2.06862593 ... -1.66986678 -1.66114963
 -1.53582845]

predicted_errorz: 
[-1.03750438 -0.08374549 -1.26147564 ... -1.45418374 -1.22540578
 -1.48669444]



In [101]:
submission_ecef = pd.DataFrame()
submission_ecef['MeasurementX_Corr'] = merge_test_df['WlsPositionXEcefMeters'] + predicted_errorx
submission_ecef['MeasurementY_Corr'] = merge_test_df['WlsPositionYEcefMeters'] + predicted_errory
submission_ecef['MeasurementZ_Corr'] = merge_test_df['WlsPositionZEcefMeters'] + predicted_errorz

In [105]:
transformer = Transformer.from_crs("EPSG:4978", "EPSG:4326", always_xy=True)

lon, lat, alt = transformer.transform(
    submission_ecef['MeasurementX_Corr'].values, 
    submission_ecef['MeasurementY_Corr'].values,
    submission_ecef['MeasurementZ_Corr'].values,
)

In [106]:
submission = pd.DataFrame()
submission['tripId'] = merge_test_df['tripId']
submission['UnixTimeMillis'] = merge_test_df['UnixTimeMillis']
submission['LatitudeDegrees'] = lat
submission['LongitudeDegrees'] = lon

print(submission)
print()
submission.to_csv('./SUBMISSIONS/RANDOM_FOREST/submission.csv', index=False)

                                          tripId  UnixTimeMillis  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650832999   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650833999   
2      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650834999   
3      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650835999   
4      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650836999   
...                                          ...             ...   
66092           2022-04-25-US-OAK-2/GooglePixel4   1650927742650   
66093           2022-04-25-US-OAK-2/GooglePixel4   1650927743642   
66094           2022-04-25-US-OAK-2/GooglePixel4   1650927744651   
66095           2022-04-25-US-OAK-2/GooglePixel4   1650927745640   
66096           2022-04-25-US-OAK-2/GooglePixel4   1650927746632   

       LatitudeDegrees  LongitudeDegrees  
0            37.395821       -122.102971  
1            37.395855       -122.102987  
2            37.395814       -122.102995  
3          

In [400]:
print(merge_test_df.isna().sum())

UnixTimeMillis            0
Cn0DbHz                   0
SatCount                  0
SvElevationDegrees        0
tripId                    0
TotalMotionEnergy         0
WlsPositionXEcefMeters    0
WlsPositionYEcefMeters    0
WlsPositionZEcefMeters    0
dtype: int64


In [32]:
features_test_df = merge_test_df.drop(columns=['UnixTimeMillis', 'tripId', 'WlsPositionXEcefMeters','WlsPositionYEcefMeters', 'WlsPositionZEcefMeters'])

predicted_errorx = rf_regressor_x.predict(features_test_df)
predicted_errory = rf_regressor_y.predict(features_test_df)
predicted_errorz = rf_regressor_z.predict(features_test_df)

print('predicted_errorx: ')
print(predicted_errorx)
print()
print('predicted_errory: ')
print(predicted_errory)
print()
print('predicted_errorz: ')
print(predicted_errorz)
print()

predicted_errorx: 
[ 0.4020744   0.4211344   0.44863316 ... -0.99021174 -0.97466552
 -1.06895961]

predicted_errory: 
[-0.85241053 -0.79379261 -0.7737016  ... -1.6287424  -1.62517746
 -1.69148099]

predicted_errorz: 
[1.1027246  1.07259231 0.97923335 ... 0.34618789 0.44081743 0.53964961]



In [33]:
submission_ecef = pd.DataFrame()
submission_ecef['MeasurementX_Corr'] = merge_test_df['WlsPositionXEcefMeters'] + predicted_errorx
submission_ecef['MeasurementY_Corr'] = merge_test_df['WlsPositionYEcefMeters'] + predicted_errory
submission_ecef['MeasurementZ_Corr'] = merge_test_df['WlsPositionZEcefMeters'] + predicted_errorz

In [34]:
transformer = Transformer.from_crs("EPSG:4978", "EPSG:4326", always_xy=True)

lon, lat, alt = transformer.transform(
    submission_ecef['MeasurementX_Corr'].values, 
    submission_ecef['MeasurementY_Corr'].values,
    submission_ecef['MeasurementZ_Corr'].values,
)

In [35]:
submission = pd.DataFrame()
submission['tripId'] = merge_test_df['tripId']
submission['UnixTimeMillis'] = merge_test_df['UnixTimeMillis']
submission['LatitudeDegrees'] = lat
submission['LongitudeDegrees'] = lon

print(submission)
print()
submission.to_csv('./SUBMISSIONS/RANDOM_FOREST/submission.csv', index=False)

                                          tripId  UnixTimeMillis  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650832999   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650833999   
2      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650834999   
3      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650835999   
4      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650836999   
...                                          ...             ...   
66092           2022-04-25-US-OAK-2/GooglePixel4   1650927742650   
66093           2022-04-25-US-OAK-2/GooglePixel4   1650927743642   
66094           2022-04-25-US-OAK-2/GooglePixel4   1650927744651   
66095           2022-04-25-US-OAK-2/GooglePixel4   1650927745640   
66096           2022-04-25-US-OAK-2/GooglePixel4   1650927746632   

       LatitudeDegrees  LongitudeDegrees  
0            37.395833       -122.102959  
1            37.395856       -122.102978  
2            37.395840       -122.102992  
3          